In [1]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

In [2]:
SYSTEM_PROMPT_STRING = """
Your task is to partition the 16 words into 4 groups of 4 words/phrases based on shared connections. 
Output requirements (STRICT): 
OUTPUT ONLY the final groups of words/phrases.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups.
Use ONLY the EXACT words/phrases from the puzzle.
Make sure there are EXACTLY 4 groups of 4 words/phrases each with their category names. NO EXCEPTIONS.
Return the answer exactly in this format:

GROUP 1: word1 || word2 || word3 || word4
GROUP 2: word1 || word2 || word3 || word4
GROUP 3: word1 || word2 || word3 || word4
GROUP 4: word1 || word2 || word3 || word4
"""

In [ ]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = " || ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split("||")] for group in groups ]
    
    return parsed_groups

In [4]:
from data_loader import get_train_test_split

ds_train, ds_test = get_train_test_split()
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

c:\Users\kunri\miniconda3\envs\cs175\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training puzzles: 521
Testing puzzles: 131


In [ ]:
import random

#Number of times LLM can retry if it hallucinates before marking it as an error
MAX_RETRIES = 4

def valid_prediction(pred_groups, words16):
    if pred_groups is None:
        return False
    if len(pred_groups) != 4:
        return False
    if any(len(group) != 4 for group in pred_groups):
        return False
    
    pred_words = [word for group in pred_groups for word in group]
    if set(pred_words) == set(words16):
        return True
    else:
        return False

def make_few_shot_prompt(words16, k, split_for_few_shot):
    few_shot_prompt = "Here are some previous examples:\n"
    count = 0
    for fs_row in split_for_few_shot:
        fs_words = fs_row["words"]
        # Skip because of leakage
        if set(fs_words) == set(words16):
            continue
        fs_groups = [ans["words"] for ans in fs_row["answers"]]
        fs_text = f"Here are 16 words: {' || '.join(fs_words)}\n"
        for i, group in enumerate(fs_groups, 1):
            fs_text += f"GROUP {i}: {' || '.join(group)}\n"
        few_shot_prompt += fs_text + "\n"
        count += 1
        if count >= k:
            break
    return few_shot_prompt

def solve_puzzle(words16, k=0, split_for_few_shot=None, model="llama3.1-8b", temperature=0.1, max_tokens=600, max_retries=MAX_RETRIES):
    few_shot_prompt = ""
    if k > 0 and split_for_few_shot is not None:
        few_shot_examples = random.sample(list(split_for_few_shot), k)
        few_shot_prompt = make_few_shot_prompt(words16, k, few_shot_examples)
    user_prompt = few_shot_prompt + "NOW, solve this puzzle and only this one: " + convert_puzzle_to_prompt(words16)

    attempt = 0
    while attempt <= max_retries:
        response = call_llm(
            SYSTEM_PROMPT_STRING,
            user_prompt,
            model=model,
            temperature=(temperature + 0.1*attempt),  
            max_tokens=max_tokens
        )

        pred_groups = parse_response_to_pred_groups(response)

        if valid_prediction(pred_groups, words16):
            return pred_groups

        print(f"Invalid LLM output: {response}\nCorrect Answer: {words16}\nRetrying ({attempt+1}/{max_retries})")
        attempt += 1

    print("ERROR: LLM failed after retries")
    return []

In [6]:
row0 = ds_train[0]
words16 = row0["words"]
pred_groups = solve_puzzle(words16)

print("\nPredicted groups:")
for g in pred_groups:
    print(g)

print("\nGold groups:")
for ans in row0["answers"]:
    print(ans["answerDescription"], "->", ans["words"])


Predicted groups:
['GIANT', 'JUMBO', 'MONSTER', 'SUPER']
['BOWL', 'LADLE', 'POT', 'SPOON']
['CHARACTER', 'INDIVIDUAL', 'PARTY', 'PERSON']
['ALIEN', 'AVATAR', 'DUNE', 'TRON']

Gold groups:
MASSIVE -> ['GIANT', 'JUMBO', 'MONSTER', 'SUPER']
USED WHEN SERVING SOUP -> ['BOWL', 'LADLE', 'POT', 'SPOON']
SOMEBODY -> ['CHARACTER', 'INDIVIDUAL', 'PARTY', 'PERSON']
SCI-FI FRANCHISES -> ['ALIEN', 'AVATAR', 'DUNE', 'TRON']


In [7]:
from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    evaluate,
    gold_groups_from_row,
)

N_EVAL = len(ds_test)

few_shot_values = [0, 1, 3, 5, 10, 20]
for k in few_shot_values:

    results = evaluate(
        ds_test,
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: solve_puzzle(words, k=k, split_for_few_shot=ds_train),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=True,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")

Processed 10/131 samples
Processed 20/131 samples
Invalid LLM output: GROUP 1: TEE (GOLF) || TEE (SHIRT) || GEAR || SAW
GROUP 2: TEA || TEE (SHIRT) || ZIPPER || COMB
GROUP 3: TI (MUSICAL NOTE) || GEEZ || NUTS || FUDGE
GROUP 4: RATS || BANK || DELTA || MOUTH
Correct Answer: ['TEA', 'TEE (GOLF)', 'TEE (SHIRT)', 'TI (MUSICAL NOTE)', 'COMB', 'GEAR', 'SAW', 'ZIPPER', 'FUDGE', 'GEEZ', 'NUTS', 'RATS', 'BANK', 'BED', 'DELTA', 'MOUTH']
Retrying (1/4)
Processed 30/131 samples
Processed 40/131 samples
Processed 50/131 samples
Processed 60/131 samples
Processed 70/131 samples
Processed 80/131 samples
Invalid LLM output: GROUP 1: TO || TOO || TUE || TIE
GROUP 2: COUPLE || WED || UNITE || TIE
GROUP 3: LAID || PLACED || PUT || SAT
GROUP 4: MAY || SUN || TWO || WILD
Correct Answer: ['TO', 'TOO', 'TUE', 'TWO', 'COUPLE', 'TIE', 'UNITE', 'WED', 'LAID', 'PLACED', 'PUT', 'SAT', 'MAY', 'SUN', 'WALL', 'WILD']
Retrying (1/4)
Invalid LLM output: GROUP 1: TO || TOO || TUE || TIE
GROUP 2: COUPLE || WED || UNITE 

KeyboardInterrupt: 